# Building a predictive model with filtered input data

In [1]:
%matplotlib inline

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import collections
import json

PATH = "../data/preemi-30apr2021.dta"

## Cleaning data

### Drop columns


In [2]:
df = pd.read_stata(PATH)

df[df == ""] = np.nan

columns = list(df)

# Unique columns
df = df.drop(columns=["protocolid", "personid", "pr_id", "ch_id", "mt_id"])

# Event status
df = df.drop(columns=[c for c in columns if "eventstatus_e" in c])

# Versions
df = df.drop(columns=[c for c in columns if "versionname_e" in c])

# Constant or almost unique
df = df.drop(columns=["haemounava_e1"])
df = df.drop(columns=["checklist_e1"])
df = df.drop(columns=["matstata_e2"])

# Remove columns with high NaNs
N = df.shape[0]
for col in columns:
    if col not in df: continue
    n = df[df[col].isnull()].shape[0]
    if n/N > 0.5:
        df = df.drop(columns=[col])

## Sort by date

In [3]:
df = df.sort_values(by="mensdate_e1").copy()

In [4]:
# Remove redundant _e

red_cols = collections.defaultdict(list)
for c in list(df.columns):
    x = c.split("_")
    if len(x[-1]) == 2 and x[-1][0] == "e":
        red_cols["_".join(x[:-1])] += ["_".join(x)]

def get_to_drop(v):
    b = -1
    best = None
    for x in v:
        if int(x[-1]) > b:
            b = int(x[-1])
            best = x
    return [x for x in v if x != best]

for k,v in red_cols.items():
    v = get_to_drop(v)
    #df = df.drop(columns=v)

df = df.drop(columns = ["compperson_e5"])

In [5]:
# Minor cleans

df.loc[df["facid_e5"] == "FO3", "facid_e5"] = "F03"
df.loc[df["facid_e5"] == "FO2", "facid_e5"] = "F02"

In [6]:
# Remove dirty variables

df = df.drop(columns=["BWDQ_cat", "birthweightcat1", "gestagecat2", "gestagecat1"])

# Newly realised redundant
df = df.drop(columns = ["facid_e5"])

In [7]:
# Clean data type

def datatype_cleaner(df, col):
    df_ = df.copy()
    df_ = df_[df_[col].notnull()]
    N = df_.shape[0]
    df_["numnew"] = pd.to_numeric(df_[col], errors="coerce")
    df_["datnew"] = pd.to_datetime(df_[col], errors="coerce")
    if df_[df_["numnew"].notnull()].shape[0]/N > 0.8:
        df[col] = pd.to_numeric(df[col])
        return df
    if df_[df_["datnew"].notnull()].shape[0]/N > 0.8:
        df[col] = pd.to_datetime(df[col], errors="coerce")
        df.loc[df[col] < pd.Timestamp(1910,1,1)] = np.nan
        return df
    return df

for col in columns:
    if col not in df: continue
    df = datatype_cleaner(df, col)
    

In [8]:
df.loc[df["matage"] > 60, "matage"] = np.nan
df.loc[df["eduyrs_e1"] > 15, "eduyrs_e1"] = 15
df.loc[df["matwei_e1"] > 200, "matwei_e1"] = np.nan
df.loc[df["babwght_e2"] > 8000, "babwght_e2"] = np.nan
df.loc[df["bwdat"] == "Birth weight missing", "bwdat"] = np.nan
df.loc[df["fetneo_e2"] == "3=Fresh Stillbirth (No movement", "fetneo_e2"] = "3=Fresh Stillbirth"
df[df == "Unknown"] = np.nan
df[df == "unknown"] = np.nan
df[df == "Don't know"] = np.nan

df["vitalstatus_ltf"] = np.nan
df.loc[df["vitalstatus"] == "LTF", ["vitalstatus_ltf"]] = "LTF"
df.loc[(df["vitalstatus"].notnull()) & (df["vitalstatus"] != "LTF"), ["vitalstatus_ltf"]] = "Live birth or Pregnancy loss"
df.loc[df["vitalstatus"] == "LTF", ["vitalstatus"]] = np.nan

df["pr_outcome_miscarriage"] = np.nan
df.loc[df["pr_outcome"] == "Miscarriage", ["pr_outcome_miscarriage"]] = "Miscarriage"
df.loc[(df["pr_outcome"] != "Miscarriage") & (df["pr_outcome"].notnull()), ["pr_outcome_miscarriage"]] = "No miscarriage"
df["pr_outcome_alive"] = np.nan
df.loc[df["pr_outcome"].isin(["Miscarriage", "Stillbirth"]), ["pr_outcome_alive"]] = "Miscarriage or Stillbirth"
df.loc[df["pr_outcome"] == "Live birth", ["pr_outcome_alive"]] = "Live birth"
df = df.drop(columns=["pr_outcome"])

df.to_csv("df.csv", index=False)
df = pd.read_csv("df.csv")

## Filtered based on Kati and Albert's input

In [9]:
outcomes = {
    "Pregnancy Outcome": ["vitalstatus28", "still",
                          "vitalstatus", "vitalstatus_ltf",  "livebirth", "pr_outcome_miscarriage", "pr_outcome_alive"],
    "Early Neonatal Death": ["infsta_e3"],
    "Late Neonatal Death": ["infsta_e4"],
    "Pre-term Delivery": ["babterm_e2", "preterm"]
}

gestation = {
    "Gestation": ["g_age", "gestage"],
    "Expected Due Date": ["edd_e1", "estedd_e1"],
    "Last Menstrual Period": ["mensdate_e1"],
    "Delivery Date": ["delidate1_e1", "motconpl_e2"],
    "Method of Determining Gestation": ["gestmethod"]
}

counfounders = {
    "Maternal Age": ["matage", "age_cat", "matagecat", "dob_day"],
    "School Level": ["schlev_e1"],
    "Years of Education": ["eduyrs_e1"],
    "Parity": ["parity_lbsb", "parity_cat", "parity"], # Number of viable births
    "BMI": ["mathei_e1", "matwei_e1"],
    "Antenatal Visits": ["antcarfreq_e2", "anc_visit"],
    "Delivery By": ["delivby_e2"],
    "Delivery Place": ["delivwhr_e2", "delivfac_e2", "deplace"],
    "Mode of Delivery": ["mod_e2", "detype"],
    "Baby Sex": ["babysex_e2", "sex"],
    "Multiple Birth": ["multbirth_e2", "multiple"],
    "Birthweight Determined": ["bwdat"], # remove
    "Birthweight": ["birthweight"],
    "Gravida": ["gravida_e1"], # Number of pregnancies (remove)? Counfounder / Is not an intervention
    "Neonatal Abx": ["neotreant_e2"], # Same
}

neonatal_tx = {
    "Neonatal Antibiotics": ["neotreant_e2"],
    "CPAP": ["neotrecpap_e2"],
    "Oxygen": ["neotreoxy_e2"]
}

interventions = {
    "Antenatal Visits": ["antcarfreq_e2"],
    "Dexamethasone": ["mattredex_e2"],
    "Kangaroo Mother Care Skin to Skin": ["neotrekmc_e2"],
    "Cord care Chlorhexidine": ["neotremcc_e2"],
    "Bag and Mask Resuscitation": ["babresbm_e2"],
}

all_variables = dict((k,v) for d in [outcomes, gestation, counfounders, neonatal_tx, interventions] for k,v in d.items())

all_columns = list(set([x for k,v in all_variables.items() for x in v]))

dt = df[all_columns]

In [10]:
variable_stages = {
    "Day0": [
        "Maternal Age",
        "School Level",
        "Years of Education",
        "Parity",
        "Gravida",
        "BMI",
        "Multiple Birth",
    ],
    "Pregnancy": [
        "Antenatal Visits"
    ],
    "Delivery": [
        "Birthweight Determined",
        "Birthweight",
        "Method of Determining Gestation",
        "Last Menstrual Period",
        "Baby Sex",
        "Dexamethasone",
        "CPAP",
        "Oxygen",
        "Kangaroo Mother Care Skin to Skin",
        "Cord care Chlorhexidine",
        "Bag and Mask Resuscitation",
        "Delivery Date"
    ]
}


In [11]:
from matplotlib.dates import date2num
from pandas.api.types import is_datetime64_any_dtype as is_datetime

def is_date_column(df, c):
    if is_datetime(df[df[c].notnull()][c]):
        return True
    else:
        return False

def convert_date_to_num(x):
    try:
        return date2num(x)
    except:
        return np.nan
    
dt = dt.copy()
for c in list(dt.columns):
    if is_date_column(dt, c):
        dt.loc[:, [c]] = [convert_date_to_num(x) for x in list(dt[c])]
        
def binarize_variable(dt, c, t):
    b = c + "_bin"
    dt[b] = np.nan
    dt.loc[dt[c] == t, [b]] = 1
    dt.loc[(dt[c] != t) & (dt[c].notnull()), [b]] = 0
    dt = dt.drop(columns = [c])
    dt = dt.rename(columns={b: c})
    return dt

dt = binarize_variable(dt, "vitalstatus28", "Dead")
dt = binarize_variable(dt, "vitalstatus", "Pregnancy loss")
dt = binarize_variable(dt, "vitalstatus_ltf", "LTF")
dt = binarize_variable(dt, "livebirth", "Pregnancy loss")
dt = binarize_variable(dt, "pr_outcome_miscarriage", "Miscarriage")
dt = binarize_variable(dt, "pr_outcome_alive", "Miscarriage or Stillbirth")
dt = binarize_variable(dt, "infsta_e3", "Dead")
dt = binarize_variable(dt, "infsta_e4", "Dead")
dt = binarize_variable(dt, "preterm", "Preterm")

In [12]:
from autogluon.tabular import TabularDataset, TabularPredictor
from autogluon.core.utils.utils import setup_outputdir
from autogluon.core.utils.loaders import load_pkl
from autogluon.core.utils.savers import save_pkl
import os.path

class MultilabelPredictor():
    """ Tabular Predictor for predicting multiple columns in table.
        Creates multiple TabularPredictor objects which you can also use individually.
        You can access the TabularPredictor for a particular label via: `multilabel_predictor.get_predictor(label_i)`

        Parameters
        ----------
        labels : List[str]
            The ith element of this list is the column (i.e. `label`) predicted by the ith TabularPredictor stored in this object.
        path : str
            Path to directory where models and intermediate outputs should be saved.
            If unspecified, a time-stamped folder called "AutogluonModels/ag-[TIMESTAMP]" will be created in the working directory to store all models.
            Note: To call `fit()` twice and save all results of each fit, you must specify different `path` locations or don't specify `path` at all.
            Otherwise files from first `fit()` will be overwritten by second `fit()`.
            Caution: when predicting many labels, this directory may grow large as it needs to store many TabularPredictors.
        problem_types : List[str]
            The ith element is the `problem_type` for the ith TabularPredictor stored in this object.
        eval_metrics : List[str]
            The ith element is the `eval_metric` for the ith TabularPredictor stored in this object.
        consider_labels_correlation : bool
            Whether the predictions of multiple labels should account for label correlations or predict each label independently of the others.
            If True, the ordering of `labels` may affect resulting accuracy as each label is predicted conditional on the previous labels appearing earlier in this list (i.e. in an auto-regressive fashion).
            Set to False if during inference you may want to individually use just the ith TabularPredictor without predicting all the other labels.
        kwargs :
            Arguments passed into the initialization of each TabularPredictor.

    """

    multi_predictor_file = 'multilabel_predictor.pkl'

    def __init__(self, labels, path, problem_types=None, eval_metrics=None, consider_labels_correlation=False, **kwargs):
        if len(labels) < 2:
            raise ValueError("MultilabelPredictor is only intended for predicting MULTIPLE labels (columns), use TabularPredictor for predicting one label (column).")
        self.path = setup_outputdir(path, warn_if_exist=False)
        self.labels = labels
        self.consider_labels_correlation = consider_labels_correlation
        self.predictors = {}  # key = label, value = TabularPredictor or str path to the TabularPredictor for this label
        if eval_metrics is None:
            self.eval_metrics = {}
        else:
            self.eval_metrics = {labels[i] : eval_metrics[i] for i in range(len(labels))}
        problem_type = None
        eval_metric = None
        for i in range(len(labels)):
            label = labels[i]
            path_i = self.path + "Predictor_" + label
            if problem_types is not None:
                problem_type = problem_types[i]
            if eval_metrics is not None:
                eval_metric = self.eval_metrics[label]
            self.predictors[label] = TabularPredictor(label=label, problem_type=problem_type, eval_metric=eval_metric, path=path_i, **kwargs)

    def fit(self, train_data, tuning_data=None, **kwargs):
        """ Fits a separate TabularPredictor to predict each of the labels.

            Parameters
            ----------
            train_data, tuning_data : str or autogluon.tabular.TabularDataset or pd.DataFrame
                See documentation for `TabularPredictor.fit()`.
            kwargs :
                Arguments passed into the `fit()` call for each TabularPredictor.
        """
        if isinstance(train_data, str):
            train_data = TabularDataset(train_data)
        if tuning_data is not None and isinstance(tuning_data, str):
            tuning_data = TabularDataset(tuning_data)
        train_data_og = train_data.copy()
        if tuning_data is not None:
            tuning_data_og = tuning_data.copy()
        else:
            tuning_data_og = None
        save_metrics = len(self.eval_metrics) == 0
        for i in range(len(self.labels)):
            label = self.labels[i]
            predictor = self.get_predictor(label)
            if not self.consider_labels_correlation:
                labels_to_drop = [l for l in self.labels if l != label]
            else:
                labels_to_drop = [self.labels[j] for j in range(i+1, len(self.labels))]
            train_data = train_data_og.drop(labels_to_drop, axis=1)
            if tuning_data is not None:
                tuning_data = tuning_data_og.drop(labels_to_drop, axis=1)
            print(f"Fitting TabularPredictor for label: {label} ...")
            predictor.fit(train_data=train_data, tuning_data=tuning_data, **kwargs)
            self.predictors[label] = predictor.path
            if save_metrics:
                self.eval_metrics[label] = predictor.eval_metric
        self.save()

    def predict(self, data, **kwargs):
        """ Returns DataFrame with label columns containing predictions for each label.

            Parameters
            ----------
            data : str or autogluon.tabular.TabularDataset or pd.DataFrame
                Data to make predictions for. If label columns are present in this data, they will be ignored. See documentation for `TabularPredictor.predict()`.
            kwargs :
                Arguments passed into the predict() call for each TabularPredictor.
        """
        return self._predict(data, as_proba=False, **kwargs)

    def predict_proba(self, data, **kwargs):
        """ Returns dict where each key is a label and the corresponding value is the `predict_proba()` output for just that label.

            Parameters
            ----------
            data : str or autogluon.tabular.TabularDataset or pd.DataFrame
                Data to make predictions for. See documentation for `TabularPredictor.predict()` and `TabularPredictor.predict_proba()`.
            kwargs :
                Arguments passed into the `predict_proba()` call for each TabularPredictor (also passed into a `predict()` call).
        """
        return self._predict(data, as_proba=True, **kwargs)

    def evaluate(self, data, **kwargs):
        """ Returns dict where each key is a label and the corresponding value is the `evaluate()` output for just that label.

            Parameters
            ----------
            data : str or autogluon.tabular.TabularDataset or pd.DataFrame
                Data to evalate predictions of all labels for, must contain all labels as columns. See documentation for `TabularPredictor.evaluate()`.
            kwargs :
                Arguments passed into the `evaluate()` call for each TabularPredictor (also passed into the `predict()` call).
        """
        data = self._get_data(data)
        eval_dict = {}
        for label in self.labels:
            print(f"Evaluating TabularPredictor for label: {label} ...")
            predictor = self.get_predictor(label)
            data_ = data[data[label].notnull()]
            eval_dict[label] = predictor.evaluate(data_, **kwargs)
            if self.consider_labels_correlation:
                data[label] = predictor.predict(data, **kwargs)
        return eval_dict

    def save(self):
        """ Save MultilabelPredictor to disk. """
        for label in self.labels:
            if not isinstance(self.predictors[label], str):
                self.predictors[label] = self.predictors[label].path
        save_pkl.save(path=self.path+self.multi_predictor_file, object=self)
        print(f"MultilabelPredictor saved to disk. Load with: MultilabelPredictor.load('{self.path}')")

    @classmethod
    def load(cls, path):
        """ Load MultilabelPredictor from disk `path` previously specified when creating this MultilabelPredictor. """
        path = os.path.expanduser(path)
        if path[-1] != os.path.sep:
            path = path + os.path.sep
        return load_pkl.load(path=path+cls.multi_predictor_file)

    def get_predictor(self, label):
        """ Returns TabularPredictor which is used to predict this label. """
        predictor = self.predictors[label]
        if isinstance(predictor, str):
            return TabularPredictor.load(path=predictor)
        return predictor

    def _get_data(self, data):
        if isinstance(data, str):
            return TabularDataset(data)
        return data.copy()

    def _predict(self, data, as_proba=False, **kwargs):
        data = self._get_data(data)
        if as_proba:
            predproba_dict = {}
        for label in self.labels:
            print(f"Predicting with TabularPredictor for label: {label} ...")
            predictor = self.get_predictor(label)
            if as_proba:
                predproba_dict[label] = predictor.predict_proba(data, as_multiclass=True, **kwargs)
            data[label] = predictor.predict(data, **kwargs)
        if not as_proba:
            return data[self.labels]
        else:
            return predproba_dict

In [13]:
labels = [x for k,v in outcomes.items() for x in v]
problem_types = []
for l in labels:
    labs = set([x for x in list(dt[l]) if str(x) != "nan"])
    if len(labs) == 2:
        problem_types += ["binary"]
    else:
        problem_types += ["multiclass"]
eval_metrics = ["f1"]*len(problem_types)
time_limit = 60

In [14]:
variable_stages = {
    "Day0": [
        "Maternal Age",
        "School Level",
        "Years of Education",
        "Parity",
        "Gravida",
        "BMI",
        "Multiple Birth",
    ],
    "Pregnancy": [
        "Antenatal Visits"
    ],
    "Delivery": [
        "Birthweight Determined",
        "Birthweight",
        "Method of Determining Gestation",
        "Last Menstrual Period",
        "Baby Sex",
        "Dexamethasone",
        "CPAP",
        "Oxygen",
        "Kangaroo Mother Care Skin to Skin",
        "Cord care Chlorhexidine",
        "Bag and Mask Resuscitation",
        "Delivery Date"
    ]
}


def filter_by_stage(df, stage):
    stage_columns = []
    if stage == "Day0":
        cats = variable_stages["Day0"]
    if stage == "Pregnancy":
        cats = variable_stages["Day0"] + variable_stages["Pregnancy"]
    if stage == "Delivery":
        cats = variable_stages["Day0"] + variable_stages["Pregnancy"] + variable_stages["Delivery"]
    stage_columns = []
    for k,v in all_variables.items():
        if k in cats:
            for x in v:
                stage_columns += [x]
    columns = []
    for c in list(df.columns):
        if c in labels:
            columns += [c]
        if c in stage_columns:
            columns += [c]
    return df[columns]


def ml(dt, stage):
    save_path = 'model-{0}'.format(stage)
    dt_ = TabularDataset(filter_by_stage(dt, stage))
    n = dt_.shape[0]
    train_size = 0.5
    n_tr = int(n*train_size)
    n_te = n - n_tr
    data_train = dt_.head(n_tr)
    data_test = dt_.tail(n_te)
    multi_predictor = MultilabelPredictor(labels=labels, problem_types=problem_types, path=save_path, eval_metrics=eval_metrics)
    multi_predictor.fit(data_train, time_limit=time_limit)
    performance = multi_predictor.evaluate(data_test)
    data = {
        "features": [x for x in list(dt_.columns) if x not in labels],
        "performance": performance
    }
    with open(save_path+".json", "w") as f:
        json.dump(data, f, indent=4)
    return performance



In [15]:
for stage in ["Day0", "Pregnancy", "Delivery"]:
    ml(dt, stage)

Beginning AutoGluon training ... Time limit = 60s
AutoGluon will save models to "model-Day0/Predictor_vitalstatus28/"
AutoGluon Version:  0.3.1
Train Data Rows:    5767
Train Data Columns: 14
Preprocessing data ...
Selected class <--> label mapping:  class 1 = 1, class 0 = 0
Using Feature Generators to preprocess the data ...
Fitting AutoMLPipelineFeatureGenerator...
	Available Memory:                    21392.92 MB
	Train Data (Original)  Memory Usage: 2.12 MB (0.0% of available memory)
	Inferring data type of each feature based on column values. Set feature_metadata_in to manually specify special dtypes of the features.
	Stage 1 Generators:
		Fitting AsTypeFeatureGenerator...
	Stage 2 Generators:
		Fitting FillNaFeatureGenerator...
	Stage 3 Generators:
		Fitting IdentityFeatureGenerator...
		Fitting CategoryFeatureGenerator...
			Fitting CategoryMemoryMinimizeFeatureGenerator...
	Stage 4 Generators:
		Fitting DropUniqueFeatureGenerator...
	Types of features in original data (raw dtyp

Fitting TabularPredictor for label: vitalstatus28 ...


	0.0	 = Validation score   (f1)
	0.01s	 = Training   runtime
	0.11s	 = Validation runtime
Fitting model: LightGBMXT ... Training model for up to 59.71s of the 59.71s of remaining time.
/home/mduranfrigola/miniconda3/envs/preemi/lib/python3.7/site-packages/lightgbm/engine.py:239: UserWarning: 'verbose_eval' argument is deprecated and will be removed in a future release of LightGBM. Pass 'log_evaluation()' callback via 'callbacks' argument instead.
  _log_warning("'verbose_eval' argument is deprecated and will be removed in a future release of LightGBM. "
	0.0	 = Validation score   (f1)
	0.58s	 = Training   runtime
	0.0s	 = Validation runtime
Fitting model: LightGBM ... Training model for up to 59.12s of the 59.12s of remaining time.
/home/mduranfrigola/miniconda3/envs/preemi/lib/python3.7/site-packages/lightgbm/engine.py:239: UserWarning: 'verbose_eval' argument is deprecated and will be removed in a future release of LightGBM. Pass 'log_evaluation()' callback via 'callbacks' argument i

[23:07:59] WARNING: ../src/learner.cc:1095: Starting in XGBoost 1.3.0, the default evaluation metric used with the objective 'binary:logistic' was changed from 'error' to 'logloss'. Explicitly set eval_metric if you'd like to restore the old behavior.


	0.0	 = Validation score   (f1)
	3.46s	 = Training   runtime
	0.12s	 = Validation runtime
Fitting model: LightGBMLarge ... Training model for up to 52.44s of the 52.44s of remaining time.
/home/mduranfrigola/miniconda3/envs/preemi/lib/python3.7/site-packages/lightgbm/engine.py:239: UserWarning: 'verbose_eval' argument is deprecated and will be removed in a future release of LightGBM. Pass 'log_evaluation()' callback via 'callbacks' argument instead.
  _log_warning("'verbose_eval' argument is deprecated and will be removed in a future release of LightGBM. "
	0.0	 = Validation score   (f1)
	0.82s	 = Training   runtime
	0.01s	 = Validation runtime
Fitting model: WeightedEnsemble_L2 ... Training model for up to 59.95s of the 50.74s of remaining time.
	0.0	 = Validation score   (f1)
	0.6s	 = Training   runtime
	0.0s	 = Validation runtime
AutoGluon training complete, total runtime = 9.87s ...
TabularPredictor saved. To load, use: predictor = TabularPredictor.load("model-Day0/Predictor_vitals

...

Evaluation: f1 on test data: 0.3776223776223776
Evaluations on test data:
{
    "f1": 0.3776223776223776,
    "accuracy": 0.9838709677419355,
    "balanced_accuracy": 0.6525025555717814,
    "mcc": 0.3790851599481443,
    "roc_auc": 0.7487761827059218,
    "precision": 0.48214285714285715,
    "recall": 0.3103448275862069
}


Evaluating TabularPredictor for label: infsta_e3 ...


Evaluation: f1 on test data: 0.056338028169014086
Evaluations on test data:
{
    "f1": 0.056338028169014086,
    "accuracy": 0.9876634137359602,
    "balanced_accuracy": 0.5162177426083162,
    "mcc": 0.06822627173983276,
    "roc_auc": 0.7452588613565914,
    "precision": 0.15384615384615385,
    "recall": 0.034482758620689655
}
/home/mduranfrigola/miniconda3/envs/preemi/lib/python3.7/site-packages/sklearn/metrics/_classification.py:873: RuntimeWarning: invalid value encountered in double_scalars
  mcc = cov_ytyp / np.sqrt(cov_ytyt * cov_ypyp)
/home/mduranfrigola/miniconda3/envs/preemi/lib/python3.7/site-packages/sklearn/metrics/_classification.py:1248: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
Evaluation: f1 on test data: 0.0
Evaluations on test data:
{
    "f1": 0.0,
    "accuracy": 0.9975764354958986,
    "balanced_

Evaluating TabularPredictor for label: infsta_e4 ...
Evaluating TabularPredictor for label: babterm_e2 ...
Evaluating TabularPredictor for label: preterm ...


Evaluation: f1 on test data: 0.2663736263736264
Evaluations on test data:
{
    "f1": 0.2663736263736264,
    "accuracy": 0.6965454545454546,
    "balanced_accuracy": 0.550964915371695,
    "mcc": 0.14913231484006728,
    "roc_auc": 0.6246316720892992,
    "precision": 0.4967213114754098,
    "recall": 0.18198198198198198
}
